Feature Engineering
Objective
The objective of feature engineering is to transform raw customer data into a format that improves machine learning model performance.

Feature engineering will be performed only after the data has been split into training and testing datasets to prevent data leakage.

Train-Test Split Strategy
The dataset will be divided into:

Training Set (80%)
Test Set (20%)
The test set will remain untouched until final model evaluation.

Planned Feature Transformations
Missing Value Treatment
Missing values will be identified and handled using:

Mean imputation for numerical variables
Most frequent value imputation for categorical variables
Categorical Encoding
Categorical variables will be converted into machine-readable features using:

One-Hot Encoding
Examples:

Contract Type
Internet Service
Payment Method
Feature Scaling
Numerical features will be standardized using:

StandardScaler
Examples:

Tenure
Monthly Charges
Total Charges
Feature Creation
Potential engineered features may include:

Average Monthly Spend
Customer Lifetime Value Indicators
Service Count
Contract Risk Indicators
Feature Selection
Feature importance analysis will be performed to identify the most useful variables.

Methods may include:

Correlation analysis
Model-based feature importance
Recursive feature elimination
Dimensionality Reduction
Dimensionality reduction techniques may be evaluated if required.

Potential techniques:

Principal Component Analysis (PCA)
PCA will only be used if it improves model performance and reduces complexity.

Expected Output
The final output of feature engineering will be a clean feature matrix suitable for machine learning model training and evaluation.

In [46]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

print("Feature Engineering Plan Ready")


Feature Engineering Plan Ready


Load Dataset
The customer churn dataset is loaded into a working DataFrame for feature engineering and preprocessing.

A separate working copy of the dataset will be created to preserve the original data throughout the transformation process.

In [47]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

print(df.shape)

(7043, 21)


Create Working Dataset
A copy of the original dataset is created so that transformations can be performed without modifying the source data.

In [48]:
df_clean = df.copy()

print(df_clean.shape)

(7043, 21)


TotalCharges Data Cleaning
During the Data Understanding phase, the TotalCharges feature was identified as a potential data quality issue.

Although the variable represents a monetary value, it is currently stored as a string and contains records that cannot be converted directly to a numerical format.

The first step is to convert the column to a numerical data type and identify problematic records.

In [49]:
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce")

In [50]:
df_clean["TotalCharges"].isnull().sum()

np.int64(11)

Investigate Invalid TotalCharges Records
Before deciding how to handle the 11 invalid values, the affected records will be examined.

Understanding why these values are missing helps determine the most appropriate treatment strategy and avoids introducing unnecessary bias into the dataset.

In [51]:
df_clean[
    df_clean["TotalCharges"].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


Findings
The investigation reveals a consistent pattern across all 11 records.

Key Observations
All affected customers have a tenure value of 0 months.
All affected customers have missing values in the TotalCharges field.
These customers have only recently joined the service and therefore have not accumulated any total charges.
Business Interpretation
The missing values do not appear to be data entry errors.

Instead, they represent legitimate business cases where customers have not yet been billed due to having a tenure of zero months.

Treatment Decision
Because only 11 records are affected out of 7,043 total observations (approximately 0.16% of the dataset), these records will be removed.

Removing these records will:

Simplify modelling and preprocessing
Eliminate invalid numerical values
Have a negligible impact on the overall dataset
This approach allows the TotalCharges feature to be treated as a fully numerical variable during model development.

Remove Invalid Records
The 11 records containing invalid TotalCharges values will be removed from the working dataset.

Following their removal, the dataset will contain only valid numerical values for the TotalCharges feature.

In [52]:
df_clean = df_clean.dropna(subset=["TotalCharges"])

In [53]:
df_clean.shape

(7032, 21)

Findings
The dataset now contains 7,032 observations and 21 variables.

All records containing invalid TotalCharges values have been removed.

The remaining dataset contains only valid numerical values for the TotalCharges feature and is ready for further preprocessing and feature engineering.

Given that only 11 records were removed (approximately 0.16% of the original dataset), the impact on the overall dataset is negligible.

Remove Customer Identifier
The customerID feature is a unique identifier for each customer.

Because identifiers do not contain predictive information, this feature will be removed before model development.

Removing identifier columns helps prevent the model from learning patterns that are not generalisable to new customers.

In [54]:
df_clean = df_clean.drop("customerID", axis=1)

df_clean.shape

(7032, 20)

Findings
The customer identifier has been removed from the dataset.

The remaining features consist exclusively of customer characteristics, service information, billing attributes, and the target variable.

The dataset now contains 20 variables.

Define Features and Target Variable
Machine learning models require the target variable to be separated from the predictor variables.

The target variable for this project is:

Churn
All remaining variables will be treated as candidate predictors.

In [55]:
X = df_clean.drop("Churn", axis=1)

y = df_clean["Churn"]

print("Feature Matrix Shape:", X.shape)
print("Target Vector Shape:", y.shape)

Feature Matrix Shape: (7032, 19)
Target Vector Shape: (7032,)


Create Training and Testing Datasets
The dataset will now be divided into training and testing subsets.

Split Strategy
Training Set: 80%
Testing Set: 20%
The split is performed before any preprocessing or feature engineering steps to prevent data leakage and ensure a realistic evaluation of model performance.

Stratification will be applied to preserve the original churn distribution within both datasets.

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (5625, 19)
Testing Features: (1407, 19)
Training Target: (5625,)
Testing Target: (1407,)


In [57]:
numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"]

categorical_features = [
    col
    for col in X.columns
    if col not in numerical_features]

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


Build Preprocessing Pipelines
Different feature types require different preprocessing strategies.

Numerical Features
Numerical variables will be:

Standardised using StandardScaler
Categorical Features
Categorical variables will be:

Encoded using One-Hot Encoding
Using preprocessing pipelines ensures that transformations are applied consistently and reduces the risk of data leakage.

This approach also supports reproducibility and simplifies model deployment.

In [58]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

Numerical Pipeline
The numerical preprocessing pipeline standardises numerical features so that they operate on a comparable scale.

Scaling is particularly important for algorithms such as:

Logistic Regression
Support Vector Machines
K-Nearest Neighbours

In [59]:
numerical_transformer = Pipeline(
    steps=[("scaler", StandardScaler())])

numerical_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


Categorical Pipeline
Categorical features must be transformed into numerical representations before machine learning models can use them.

One-Hot Encoding creates binary indicator variables for each category while avoiding assumptions about ordinal relationships between categories.

In [60]:
categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

categorical_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one 

Create Column Transformer
The ColumnTransformer applies different preprocessing pipelines to different feature groups.

This allows numerical and categorical features to be processed appropriately within a single reusable workflow.

In [61]:
preprocessor = ColumnTransformer(transformers=[("num", numerical_transformer, numerical_features), ("cat", categorical_transformer, categorical_features)])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

Test Preprocessing Pipeline
Before building machine learning models, the preprocessing pipeline will be fitted on the training data to verify that all transformations execute successfully.

This step confirms that the dataset is fully prepared for model development.

In [62]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (5625, 45)
Processed Testing Shape: (1407, 45)


Findings
The preprocessing pipeline was successfully applied to both the training and testing datasets.

Key Observations
The training dataset contains 5,625 observations and 45 transformed features.
The testing dataset contains 1,407 observations and 45 transformed features.
The number of features increased from 19 to 45 due to One-Hot Encoding of categorical variables.
Feature Transformations
The preprocessing pipeline performed the following transformations:

Numerical Features
Standard Scaling
Feature standardisation
Categorical Features
One-Hot Encoding
Creation of binary indicator variables
Benefits
The use of preprocessing pipelines provides:

Reproducibility
Consistency across datasets
Protection against data leakage
Simplified model deployment
Reusable feature engineering workflows
The dataset is now fully prepared for machine learning model development and evaluation.

Summary and Key Findings
The feature engineering phase successfully transformed the raw customer churn dataset into a format suitable for machine learning.

Data Cleaning
The following data quality issues were addressed:

The TotalCharges feature was converted from a string to a numerical variable.
Eleven invalid records were identified and removed.
The customerID identifier was removed because it does not provide predictive value.
Dataset Preparation
The cleaned dataset contains:

7,032 customer records
20 variables
After separating the target variable:

Feature Matrix: 7,032 × 19
Target Vector: 7,032 observations
Train-Test Split
The dataset was divided into:

Training Set: 80%
Testing Set: 20%
Stratification was used to preserve the original churn distribution.

Feature Engineering
Numerical Features
SeniorCitizen
tenure
MonthlyCharges
TotalCharges
Transformations:

StandardScaler
Categorical Features
Fifteen categorical variables were identified and transformed using:

One-Hot Encoding
Preprocessing Pipeline
A reusable preprocessing pipeline was created using:

Pipeline
ColumnTransformer
StandardScaler
OneHotEncoder
The pipeline successfully transformed the data into a machine-learning-ready format.

Final Output
The preprocessing pipeline generated:

Training Matrix: 5,625 × 45
Testing Matrix: 1,407 × 45
The data is now prepared for machine learning model development.

Next Steps
The next notebook will focus on baseline model development.

The objectives will include:

Building a Logistic Regression baseline model
Generating churn predictions
Evaluating model performance
Establishing benchmark metrics for future model comparisons